In [1]:
import sys
sys.path.append('../')

from dotenv import load_dotenv
load_dotenv('../.env')

import os
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

API_BASE = "http://localhost:8000"

print(f"✅ Ready")
print(f"✅ API: {API_BASE}")

# Test API is running
try:
    r = requests.get(f"{API_BASE}/health")
    print(f"✅ API health: {r.json()}")
except:
    print("❌ API not running — start it with: uvicorn api.main:app --reload --port 8000")

✅ Ready
✅ API: http://localhost:8000
✅ API health: {'status': 'healthy', 'model': 'PopularityNet', 'version': '1.0.0'}


In [2]:
from langchain.tools import tool
from src.analytics import (
    get_top_genres, get_top_artists, find_similar_tracks,
    get_genre_mood_profile, simulate_playlist
)
import requests
import warnings
warnings.filterwarnings('ignore')

API_BASE = "http://localhost:8000"

@tool
def tool_get_top_genres() -> str:
    """Returns top music genres ranked by average popularity. Use when asked about popular genres or genre rankings."""
    df = get_top_genres(10)
    result = "Top 10 genres by popularity:\n"
    for _, row in df.iterrows():
        result += f"  {row['track_genre'].title():25s} popularity: {row['avg_popularity']:.1f} | energy: {row['avg_energy']:.2f} | danceability: {row['avg_danceability']:.2f}\n"
    return result

@tool
def tool_get_top_artists() -> str:
    """Returns top artists ranked by average popularity. Use when asked about popular artists or best performers."""
    df = get_top_artists(10)
    result = "Top 10 artists by popularity:\n"
    for _, row in df.iterrows():
        result += f"  {str(row['artists'])[:35]:35s} popularity: {row['avg_popularity']:.1f} | tracks: {int(row['track_count'])}\n"
    return result

@tool
def tool_find_similar_tracks(track_name: str = "Blinding Lights") -> str:
    """Finds songs similar to a given track using ML cosine similarity. Use for recommendations."""
    try:
        r = requests.post(f"{API_BASE}/recommend",
                         json={"track_name": track_name, "n": 5})
        data = r.json()
        if "recommendations" not in data:
            return f"Track '{track_name}' not found"
        result = f"Tracks similar to '{track_name}':\n"
        for rec in data["recommendations"]:
            result += f"  {rec['track_name'][:35]:35s} by {str(rec['artists'])[:25]:25s} | genre: {rec['track_genre']} | popularity: {rec['popularity']}\n"
        return result
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def tool_get_genre_mood(genre: str = "pop") -> str:
    """Returns mood analysis and audio profile of a music genre. Use when asked about genre characteristics."""
    try:
        r = requests.get(f"{API_BASE}/genre-profile/{genre}")
        if r.status_code == 404:
            return f"Genre '{genre}' not found"
        data = r.json()
        return (f"Genre: {data['genre'].title()}\n"
                f"Mood: {data['mood']}\n"
                f"Avg Popularity: {data['avg_popularity']}\n"
                f"Energy: {data['avg_energy']} | Valence: {data['avg_valence']} | Danceability: {data['avg_danceability']}\n"
                f"Track count: {data['track_count']}\n"
                f"Top artists: {', '.join(data['top_artists'][:3])}")
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def tool_predict_popularity(
    danceability: float = 0.5,
    energy: float = 0.7,
    loudness: float = -6.0,
    valence: float = 0.5,
    tempo: float = 120.0,
    acousticness: float = 0.1,
    genre_avg_popularity: float = 47.0,
    artist_avg_popularity: float = 50.0
) -> str:
    """Predicts how popular a track will be using neural network. Use for popularity prediction questions."""
    try:
        payload = {
            "danceability": float(danceability),
            "energy": float(energy),
            "loudness": float(loudness),
            "speechiness": 0.05,
            "acousticness": float(acousticness),
            "instrumentalness": 0.0,
            "liveness": 0.1,
            "valence": float(valence),
            "tempo": float(tempo),
            "genre_avg_popularity": float(genre_avg_popularity),
            "artist_avg_popularity": float(artist_avg_popularity)
        }
        r = requests.post(f"{API_BASE}/predict-popularity", json=payload)
        data = r.json()
        return (f"Predicted popularity: {data['predicted_popularity']}/100\n"
                f"Confidence range: {data['confidence_range']['low']} – {data['confidence_range']['high']}\n"
                f"Tier: {data['tier']}\n"
                f"Interpretation: {data['interpretation']}")
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def tool_simulate_playlist(track_names: str = "Blinding Lights,Shape of You,Bad Guy") -> str:
    """
    Runs Monte Carlo simulation to predict playlist engagement and listener drop-off.
    Pass track names as comma-separated string.
    Use when asked about playlist quality or engagement.
    """
    try:
        tracks = [t.strip() for t in track_names.split(",")]
        r = requests.post(f"{API_BASE}/simulate-playlist",
                         json={"track_names": tracks, "n_simulations": 10000})
        data = r.json()
        result = f"Playlist Simulation ({len(tracks)} tracks, 10,000 scenarios):\n"
        result += f"Full completion rate: {data['full_completion_rate']}%\n"
        result += f"Avg tracks heard: {data['avg_tracks_heard']}/{data['playlist_length']}\n"
        result += f"Verdict: {data['verdict']}\n"
        result += "Track retention:\n"
        for track, retention in zip(data['track_names'], data['track_retention']):
            bar = '█' * int(retention/10)
            result += f"  {track[:30]:30s} {retention}% {bar}\n"
        return result
    except Exception as e:
        return f"Error: {str(e)}"

tools = [
    tool_get_top_genres,
    tool_get_top_artists,
    tool_find_similar_tracks,
    tool_get_genre_mood,
    tool_predict_popularity,
    tool_simulate_playlist,
]

print(f"✅ {len(tools)} tools registered")

✅ 6 tools registered


In [9]:
import os
from dotenv import load_dotenv
load_dotenv('../.env')

from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

main_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="""You are a music intelligence AI for a streaming platform like Spotify.

You have access to real data on 114,000 tracks across 125 genres.

When answering questions:
1. Use the appropriate tools to get real data
2. Analyze the results carefully  
3. Give specific recommendations with actual numbers
4. Sound like a music industry analyst, not a chatbot

Rules:
- Never mention tool names in your response
- Always include specific numbers and artist/track names
- For playlist questions always run the simulation
- For recommendation questions always find similar tracks
- Use multiple tools when needed
- NEVER show function calls, tool names, or code in your response
- If you need more data, call the tool internally but never show it to the user
- Always write in plain English paragraphs only"""
)

reflection_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

def run_with_reflection(question: str) -> str:
    response = main_agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    initial = response['messages'][-1].content

    reflection = reflection_llm.invoke([HumanMessage(content=f"""You are a senior music industry analyst reviewing an AI recommendation.

Question: {question}
Answer: {initial}

Evaluate:
1. Does it include specific track/artist names and numbers?
2. Is the recommendation actionable?
3. Does it sound like a real music analyst?

If strong: APPROVED: [reason]
If needs improvement: IMPROVED: [better version]
Never mention tool names.""")])

    r = reflection.content
    return r.replace("IMPROVED:", "").strip() if r.startswith("IMPROVED:") else initial

print("✅ Self-reflecting music intelligence agent ready!")

✅ Self-reflecting music intelligence agent ready!


In [ ]:
def ask(question):
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print('='*60)
    answer = run_with_reflection(question)
    print(answer)

ask("What are the hottest music genres right now?")
ask("Find me songs similar to Shape of You by Ed Sheeran")
ask("What is the mood and vibe of k-pop music?")
ask("Will a high energy dance track with loud sound perform well?")
ask("Analyze this playlist: Blinding Lights, Shape of You, Bad Guy, Despacito")


Q: What are the hottest music genres right now?
Based on current trends, the top music genres ranked by average popularity are hip-hop, pop, and electronic dance music (EDM), with average popularity scores of 82, 78, and 75 respectively. Hip-hop is leading the pack, with popular artists like Drake and Kendrick Lamar consistently releasing chart-topping hits. Pop music is also thriving, with artists like Ariana Grande and Taylor Swift dominating the airwaves. EDM is another genre that's gaining traction, with festivals like Tomorrowland and Ultra Music Festival drawing in massive crowds. These genres are not only popular among younger listeners but also have a significant following across various age groups.

Q: Find me songs similar to Shape of You by Ed Sheeran
After analyzing the characteristics of "Shape of You" by Ed Sheeran, I found several songs that share similar qualities. One of the most similar tracks is "Senorita" by Shawn Mendes and Camila Cabello, which has a similar blen

: 